In [1]:
import json
import time
import re
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


class PncpTermSearchCollector:
    BASE_URL = "https://pncp.gov.br/api/search/"
    BASE_SITE_URL = "https://pncp.gov.br"

    def __init__(
        self,
        output_dir: str = "./pncp_buscas_termos",
        timeout: int = 60,
        sleep_seconds: float = 0.25,
        max_retries: int = 3,
        retry_wait: float = 2.0,
        session: Optional[requests.Session] = None,
    ) -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.max_retries = max_retries
        self.retry_wait = retry_wait

        self.session = session or requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/json",
            }
        )

    # =========================================================
    # Utilidades
    # =========================================================
    def _request_json(self, params: Dict[str, Any]) -> Dict[str, Any]:
        last_error = None

        for tentativa in range(1, self.max_retries + 1):
            try:
                response = self.session.get(
                    self.BASE_URL,
                    params=params,
                    timeout=self.timeout,
                )

                print(f"[DEBUG URL] {response.url}")

                if response.status_code == 400:
                    return {
                        "_http_400": True,
                        "items": [],
                        "total": 0,
                    }

                response.raise_for_status()
                return response.json()

            except requests.RequestException as e:
                last_error = e
                print(f"[WARN] Tentativa {tentativa}/{self.max_retries} falhou: {e}")

                if tentativa < self.max_retries:
                    time.sleep(self.retry_wait)

        raise last_error

    def _slugify(self, texto: str) -> str:
        texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
        texto = texto.lower().strip()
        texto = re.sub(r"[^a-z0-9]+", "_", texto)
        texto = re.sub(r"_+", "_", texto).strip("_")
        return texto or "termo"

    def _build_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith("http://") or item_url.startswith("https://"):
            return item_url
        return f"{self.BASE_SITE_URL}{item_url}"

    def _extract_numero_controle(self, item: Dict[str, Any]) -> Optional[str]:
        return (
            item.get("numeroControlePNCP")
            or item.get("numero_controle_pncp")
            or item.get("raw", {}).get("numero_controle_pncp")
        )

    def _extract_data_atualizacao(self, item: Dict[str, Any]) -> str:
        return (
            item.get("dataAtualizacaoPncp")
            or item.get("data_atualizacao_pncp")
            or item.get("raw", {}).get("data_atualizacao_pncp")
            or ""
        )

    # =========================================================
    # Normalização
    # =========================================================
    def _normalizar_item(self, item: Dict[str, Any], termo_busca: str) -> Dict[str, Any]:
        return {
            "termoBusca": termo_busca,
            "objetoCompra": item.get("description"),
            "titulo": item.get("title"),
            "id": item.get("id"),
            "numeroControlePNCP": item.get("numero_controle_pncp"),
            "anoCompra": item.get("ano"),
            "sequencialCompra": item.get("numero_sequencial"),
            "numero": item.get("numero"),
            "documentType": item.get("document_type"),
            "tipoId": item.get("tipo_id"),
            "tipoNome": item.get("tipo_nome"),
            "modalidadeLicitacaoId": item.get("modalidade_licitacao_id"),
            "modalidadeLicitacaoNome": item.get("modalidade_licitacao_nome"),
            "situacaoId": item.get("situacao_id"),
            "situacaoNome": item.get("situacao_nome"),
            "orgaoId": item.get("orgao_id"),
            "orgaoCnpj": item.get("orgao_cnpj"),
            "orgaoNome": item.get("orgao_nome"),
            "unidadeId": item.get("unidade_id"),
            "unidadeCodigo": item.get("unidade_codigo"),
            "unidadeNome": item.get("unidade_nome"),
            "esferaId": item.get("esfera_id"),
            "esferaNome": item.get("esfera_nome"),
            "poderId": item.get("poder_id"),
            "poderNome": item.get("poder_nome"),
            "municipioId": item.get("municipio_id"),
            "municipioNome": item.get("municipio_nome"),
            "uf": item.get("uf"),
            "dataPublicacaoPncp": item.get("data_publicacao_pncp"),
            "dataAtualizacaoPncp": item.get("data_atualizacao_pncp"),
            "dataInicioVigencia": item.get("data_inicio_vigencia"),
            "dataFimVigencia": item.get("data_fim_vigencia"),
            "dataAssinatura": item.get("data_assinatura"),
            "cancelado": item.get("cancelado"),
            "temResultado": item.get("tem_resultado"),
            "valorGlobal": item.get("valor_global"),
            "fonteOrcamentaria": item.get("fonte_orcamentaria"),
            "fonteOrcamentariaId": item.get("fonte_orcamentaria_id"),
            "fonteOrcamentariaNome": item.get("fonte_orcamentaria_nome"),
            "itemUrl": item.get("item_url"),
            "urlPncp": self._build_url(item.get("item_url")),
            "createdAt": item.get("createdAt"),
            "raw": item,
        }

    # =========================================================
    # Busca de um termo
    # =========================================================
    def buscar_termo(
        self,
        termo: str,
        tipos_documento: str = "edital",
        ordenacao: str = "-data",
        tam_pagina: int = 100,
        status: str = "recebendo_proposta",
        max_paginas: Optional[int] = None,
        deduplicar_por_id: bool = True,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        pagina = 1
        total_esperado: Optional[int] = None
        itens_brutos: List[Dict[str, Any]] = []
        itens_normalizados: List[Dict[str, Any]] = []
        ids_vistos = set()

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas atingido para '{termo}': {max_paginas}")
                break

            params = {
                "q": termo,
                "tipos_documento": tipos_documento,
                "ordenacao": ordenacao,
                "pagina": pagina,
                "tam_pagina": tam_pagina,
                "status": status,
            }

            data = self._request_json(params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] API retornou 400 para '{termo}' na página {pagina}. Encerrando.")
                break

            items = data.get("items", [])
            total = data.get("total", 0)

            if total_esperado is None:
                total_esperado = total

            if verbose:
                print(
                    f"[INFO] Termo='{termo}' | página {pagina} | "
                    f"itens={len(items)} | acumulado={len(itens_normalizados)} | total={total_esperado}"
                )

            if not items:
                break

            for bruto in items:
                if deduplicar_por_id:
                    item_id = bruto.get("id")
                    if item_id and item_id in ids_vistos:
                        continue
                    if item_id:
                        ids_vistos.add(item_id)

                itens_brutos.append(bruto)
                itens_normalizados.append(self._normalizar_item(bruto, termo))

            if len(items) < tam_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return {
            "termo": termo,
            "total_informado_api": total_esperado,
            "total_coletado": len(itens_normalizados),
            "items": itens_normalizados,
            "items_brutos": itens_brutos,
        }

    # =========================================================
    # Salvar resultado individual
    # =========================================================
    def salvar_json(self, conteudo: Any, caminho_arquivo: Path) -> None:
        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(conteudo, f, ensure_ascii=False, indent=2)

    def salvar_csv(self, items: List[Dict[str, Any]], caminho_arquivo: Path) -> None:
        df = pd.json_normalize(items)
        df.to_csv(caminho_arquivo, index=False, encoding="utf-8-sig")

    # =========================================================
    # Merge
    # =========================================================
    def merge_unicos_por_numero_controle(
        self,
        colecoes: List[Dict[str, Any]],
    ) -> Dict[str, Any]:
        unicos: Dict[str, Dict[str, Any]] = {}
        sem_chave: List[Dict[str, Any]] = []

        for colecao in colecoes:
            for item in colecao.get("items", []):
                chave = self._extract_numero_controle(item)

                if not chave:
                    sem_chave.append(item)
                    continue

                if chave not in unicos:
                    unicos[chave] = item
                else:
                    atual = unicos[chave]
                    if self._extract_data_atualizacao(item) > self._extract_data_atualizacao(atual):
                        unicos[chave] = item

        resultado_items = list(unicos.values()) + sem_chave

        return {
            "total_pesquisas": len(colecoes),
            "total_unicos_por_numero_controle_pncp": len(unicos),
            "total_sem_numero_controle_pncp": len(sem_chave),
            "total_final": len(resultado_items),
            "items": resultado_items,
        }

    # =========================================================
    # Execução completa
    # =========================================================
    def executar(
        self,
        termos: List[str],
        tipos_documento: str = "edital",
        ordenacao: str = "-data",
        tam_pagina: int = 100,
        status: str = "recebendo_proposta",
        max_paginas_por_termo: Optional[int] = None,
        salvar_csv_merge: bool = True,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        resultados_individuais: List[Dict[str, Any]] = []

        for termo in termos:
            if verbose:
                print("\n" + "=" * 80)
                print(f"[INFO] Buscando termo: {termo}")
                print("=" * 80)

            resultado = self.buscar_termo(
                termo=termo,
                tipos_documento=tipos_documento,
                ordenacao=ordenacao,
                tam_pagina=tam_pagina,
                status=status,
                max_paginas=max_paginas_por_termo,
                verbose=verbose,
            )

            slug = self._slugify(termo)
            caminho_json = self.output_dir / f"busca_{slug}.json"
            self.salvar_json(resultado, caminho_json)

            if verbose:
                print(f"[OK] JSON individual salvo em: {caminho_json}")

            resultados_individuais.append(resultado)

        merge = self.merge_unicos_por_numero_controle(resultados_individuais)

        caminho_merge_json = self.output_dir / "pncp_merge_unicos.json"
        self.salvar_json(merge, caminho_merge_json)

        if salvar_csv_merge:
            caminho_merge_csv = self.output_dir / "pncp_merge_unicos.csv"
            self.salvar_csv(merge["items"], caminho_merge_csv)
            if verbose:
                print(f"[OK] CSV merge salvo em: {caminho_merge_csv}")

        if verbose:
            print("\n" + "=" * 80)
            print("[RESUMO FINAL]")
            print("=" * 80)
            print(f"Total de pesquisas: {merge['total_pesquisas']}")
            print(f"Total únicos por numeroControlePNCP: {merge['total_unicos_por_numero_controle_pncp']}")
            print(f"Total sem chave: {merge['total_sem_numero_controle_pncp']}")
            print(f"Total final: {merge['total_final']}")
            print(f"Merge JSON: {caminho_merge_json}")

        return {
            "resultados_individuais": resultados_individuais,
            "merge": merge,
            "output_dir": str(self.output_dir),
        }

In [2]:
termos = [
    "Software de Controle de Acesso",
    "Software de segurança",
    "Software de monitoramento",
    "Software para controle de portaria",
    "Software para registro de frequência",
    "software de gestão",
    "Relógio de ponto",
    "Cartão de ponto",
    "controle de ponto",
    "reconhecimento facial",
    "registrador eletrônico de ponto biométrico",
    "Gerenciamento Eletrônico de Frequência",
    "controle de frequência",
    "Registrador de ponto facial",
    "Sistema com aplicativo facial",
    "controle de frequência facial",
    "Controle de Faces",
    "Equipamento com biometria de face",
    "Identificador Facial",
    "bastão de ronda",
    "relógio protocolador",
    "relógio cartográfico",
    "Bobina Térmica para Relógio de Ponto",
    "Papel Termosensível para Relógio de Ponto",
    "CONTROLADOR DE ACESSO",
    "Fechadura Facial",
    "fechadura biométrica",
    "Fechadura Eletrônica Facial",
    "Fechadura eletromagnética",
    "Controladora de acesso facial",
    "HID",
    "Hickvision",
    "detector de metais",
    "porta giratória",
    "Portal Detector Metal",
    "porta automática",
    "motor de portão",
    "catracas faciais",
    "catraca balcão",
    "catraca mecânica",
    "catracas eletrônicas",
    "Catraca pedestal bidirecional",
    "catraca PNE",
    "controle de acesso",
    "controle de acesso facial",
    "cancela automática",
    "cancela eletrônica",
    "leitores faciais",
    "Torniquete",
    "Conjunto Controle Acesso Área Restrita",
    "iDFace",
    "botoeira",
]

collector = PncpTermSearchCollector(
    output_dir="/Users/jose-cleiton/dev/script_pncp/pncp_buscas_termos",
    timeout=60,
    sleep_seconds=0.3,
    max_retries=3,
    retry_wait=2,
)

resultado = collector.executar(
    termos=termos,
    tipos_documento="edital",
    ordenacao="-data",
    tam_pagina=5,  # coloque 100 se quiser puxar mais por página
    status="recebendo_proposta",
    max_paginas_por_termo=None,
    salvar_csv_merge=True,
    verbose=True,
)


[INFO] Buscando termo: Software de Controle de Acesso
[DEBUG URL] https://pncp.gov.br/api/search/?q=Software+de+Controle+de+Acesso&tipos_documento=edital&ordenacao=-data&pagina=1&tam_pagina=5&status=recebendo_proposta
[INFO] Termo='Software de Controle de Acesso' | página 1 | itens=5 | acumulado=0 | total=114
[DEBUG URL] https://pncp.gov.br/api/search/?q=Software+de+Controle+de+Acesso&tipos_documento=edital&ordenacao=-data&pagina=2&tam_pagina=5&status=recebendo_proposta
[INFO] Termo='Software de Controle de Acesso' | página 2 | itens=5 | acumulado=5 | total=114
[DEBUG URL] https://pncp.gov.br/api/search/?q=Software+de+Controle+de+Acesso&tipos_documento=edital&ordenacao=-data&pagina=3&tam_pagina=5&status=recebendo_proposta
[INFO] Termo='Software de Controle de Acesso' | página 3 | itens=5 | acumulado=10 | total=114
[DEBUG URL] https://pncp.gov.br/api/search/?q=Software+de+Controle+de+Acesso&tipos_documento=edital&ordenacao=-data&pagina=4&tam_pagina=5&status=recebendo_proposta
[INFO] Te